# 02c — Training ViTPose (Small / Base / Large)

**Runtime:** V100 16GB minimum untuk Base/Large. T4 hanya feasible untuk Small.

**Prereq:** `01_dataset_conversion.ipynb` sudah selesai. MMPose sudah terinstall (dari 02b).

**Perbedaan utama ViTPose vs HRNet:**
- Backbone: Vision Transformer → butuh pretrained ViT dari ImageNet-21K
- Optimizer: AdamW dengan **layer decay** (layer lebih dalam = lr lebih kecil)
- Memory: ViTPose-Large ~2.5× lebih boros dari HRNet-W48
- Gradient checkpointing **wajib** untuk Large di V100 < 32GB

| Variant | GPU rekomen | Estimasi waktu |
|---|---|---|
| Small  | T4 16GB   | ~10–12 jam |
| Base   | V100 16GB | ~14–18 jam |
| Large  | V100 32GB | ~22–28 jam |

In [ ]:
# Cell 1: Install
import subprocess, sys
def check_mmpose():
    try:
        import mmpose; print(f'MMPose {mmpose.__version__} sudah terinstall'); return True
    except ImportError: return False
if not check_mmpose():
    !pip install torch==2.1.0 torchvision==0.16.0 --quiet
    !pip install -U openmim --quiet
    !mim install mmengine --quiet
    !mim install 'mmcv>=2.0.0' --quiet
    !mim install mmdet --quiet
    !mim install mmpose --quiet
!pip install timm>=0.9.0 --quiet
print('Dependencies ready')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Environment & Path Setup — kompatibel Kaggle DAN Colab (auto-detect)
# ══════════════════════════════════════════════════════════════════════
import os, sys

def _detect_env():
    if os.path.exists('/kaggle/working'):
        return 'kaggle'
    if os.path.exists('/content'):
        return 'colab'
    return 'local'

ENV = _detect_env()
print(f'Environment terdeteksi: {ENV}')

# ⚠️ GANTI dengan URL repo GitHub kamu sendiri (isi SEKALI saja per notebook)
GITHUB_REPO = 'https://github.com/atangorp/soccer-homography-benchmark.git'

def _link(src, dst):
    """Symlink src -> dst. Aman dipanggil berkali-kali (idempotent),
    termasuk kalau dst adalah broken symlink dari sesi sebelumnya."""
    if os.path.exists(src) and not os.path.lexists(dst):
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        os.symlink(src, dst)

if ENV == 'kaggle':
    PROJECT_ROOT = '/kaggle/working/soccer-homography-benchmark'
    if not os.path.exists(PROJECT_ROOT):
        os.system(f'git clone -q {GITHUB_REPO} "{PROJECT_ROOT}"')
    if not os.path.exists(f'{PROJECT_ROOT}/src'):
        raise RuntimeError(
            f'Git clone gagal atau repo kosong: {GITHUB_REPO}\n'
            f'Cek: (1) URL repo benar, (2) repo sudah di-push berisi folder src/,\n'
            f'(3) repo Public atau token akses sudah diset via Kaggle Secrets.'
        )
    sys.path.insert(0, PROJECT_ROOT)

    # WAJIB: attach dataset "soccer-field-raw" via "+ Add Input" di sidebar kanan
    # (ganti 'soccer-field-raw' kalau slug dataset kamu berbeda)
    _link('/kaggle/input/soccer-field-raw',
          f'{PROJECT_ROOT}/data/raw/soccer-field-localization.v9i.yolov8')
    # Output notebook 01 (COCO JSON) — attach "01-dataset-conversion" via "+ Add Input" -> Notebook Output
    _link(f'/kaggle/input/01-dataset-conversion/soccer-homography-benchmark/data/converted/annotations',
          f'{PROJECT_ROOT}/data/converted/annotations')
    # Symlink gambar asli ke struktur data/converted/images/{split} (dibutuhkan training MMPose)
    for _split in ['train', 'val', 'test']:
        _link(f'{PROJECT_ROOT}/data/raw/soccer-field-localization.v9i.yolov8/{_split}/images',
              f'{PROJECT_ROOT}/data/converted/images/{_split}')

elif ENV == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/Colab Notebooks/soccer-homography-benchmark'
    sys.path.insert(0, PROJECT_ROOT)

else:
    PROJECT_ROOT = os.getcwd()
    sys.path.insert(0, PROJECT_ROOT)

COCO_ANN_DIR = f'{PROJECT_ROOT}/data/converted/annotations'
IMAGES_DIR   = f'{PROJECT_ROOT}/data/converted/images'
import torch
for split in ['train', 'val', 'test']:
    p = f'{COCO_ANN_DIR}/{split}.json'
    print(f'  {split}.json: {"OK" if os.path.exists(p) else "TIDAK ADA"}')
if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory/1e9
    print(f'GPU: {torch.cuda.get_device_name(0)} ({vram:.1f} GB VRAM)')
    if vram < 15: print('WARN: VRAM < 15GB, hanya Small yang feasible')
    elif vram < 25: print('INFO: VRAM < 25GB, Large butuh gradient checkpointing')
print(f'PROJECT_ROOT : {PROJECT_ROOT}')


In [ ]:
# ── Pastikan semua folder output ada (wajib di Kaggle) ───────────────────────
import os
_dirs_to_create = [
    f'{PROJECT_ROOT}/artifacts/results/figures',
    f'{PROJECT_ROOT}/artifacts/results/tables',
    f'{PROJECT_ROOT}/artifacts/results/videos',
    f'{PROJECT_ROOT}/artifacts/logs/evaluation',
    f'{PROJECT_ROOT}/artifacts/logs/training',
    f'{PROJECT_ROOT}/data/converted/annotations',
    f'{PROJECT_ROOT}/data/converted/images/train',
    f'{PROJECT_ROOT}/data/converted/images/val',
    f'{PROJECT_ROOT}/data/converted/images/test',
]
for _d in _dirs_to_create:
    os.makedirs(_d, exist_ok=True)
print('Semua folder output siap')

In [ ]:
# Cell 3: Download base configs dari MMPose model zoo
CONFIG_DIR = '/tmp/vitpose_configs'
os.makedirs(CONFIG_DIR, exist_ok=True)
for v in ['small','base','large']:
    !mim download mmpose --config td-hm_ViTPose-{v}_8xb64-210e_coco-256x192 --dest {CONFIG_DIR}/
    cfg_path = f'{CONFIG_DIR}/td-hm_ViTPose-{v}_8xb64-210e_coco-256x192.py'
    print(f'ViTPose-{v}: {"OK" if os.path.exists(cfg_path) else "GAGAL download"}')

VITPOSE_PRETRAINED = {
    'small': 'https://download.openmmlab.com/mmpose/v1/body_2d_keypoint/topdown_heatmap/coco/td-hm_ViTPose-small_8xb64-210e_coco-256x192-62d7a712_20230314.pth',
    'base' : 'https://download.openmmlab.com/mmpose/v1/body_2d_keypoint/topdown_heatmap/coco/td-hm_ViTPose-base_8xb64-210e_coco-256x192-216eae50_20230314.pth',
    'large': 'https://download.openmmlab.com/mmpose/v1/body_2d_keypoint/topdown_heatmap/coco/td-hm_ViTPose-large_8xb64-210e_coco-256x192-53474cf1_20230314.pth',
}

In [ ]:
# Cell 4: Config builder — patch 17 keypoints COCO → 32 keypoints soccer
from mmengine.config import Config

def make_vitpose_config(variant, coco_ann_dir, images_dir, output_dir, use_grad_checkpoint=False):
    base = f'{CONFIG_DIR}/td-hm_ViTPose-{variant}_8xb64-210e_coco-256x192.py'
    cfg  = Config.fromfile(base)
    # Patch: jumlah keypoints
    cfg.model.head.out_channels = 32
    # Patch: dataset
    soccer_meta = dict(
        dataset_name='soccer_field', num_keypoints=32,
        keypoint_id2name={i: str(i) for i in range(32)},
        keypoint_name2id={str(i): i for i in range(32)},
        upper_body_ids=list(range(16)), lower_body_ids=list(range(16,32)),
        flip_indices=list(range(32)), skeleton_links=[],
    )
    for split in ['train_dataloader','val_dataloader','test_dataloader']:
        sname = split.replace('_dataloader','')
        cfg[split].dataset.ann_file    = f'{coco_ann_dir}/{sname}.json'
        cfg[split].dataset.data_prefix = dict(img=images_dir)
        cfg[split].dataset.metainfo    = soccer_meta
    cfg.val_evaluator.ann_file  = f'{coco_ann_dir}/val.json'
    cfg.test_evaluator.ann_file = f'{coco_ann_dir}/test.json'
    # Patch: gradient checkpointing untuk hemat VRAM
    if use_grad_checkpoint:
        cfg.model.backbone.with_cp = True
        print('  Gradient checkpointing AKTIF — training ~25% lebih lambat tapi hemat VRAM')
    # Patch: layer decay (KRITIS untuk ViTPose — jangan hapus)
    layer_decay = {'small': 0.65, 'base': 0.75, 'large': 0.80}[variant]
    cfg.optim_wrapper.optimizer.lr = 5e-4
    if hasattr(cfg.optim_wrapper, 'paramwise_cfg'):
        cfg.optim_wrapper.paramwise_cfg.layer_decay_rate = layer_decay
    # Patch: epochs, output dir
    cfg.train_cfg.max_epochs = 210
    cfg.work_dir = output_dir
    cfg.default_hooks.checkpoint.save_best = 'coco/AP'
    cfg.default_hooks.checkpoint.interval  = 10
    return cfg

print('Config builder siap')

In [ ]:
# Cell 5: TRAINING — ganti VARIANT untuk setiap run
# ⚠️ Restart runtime setelah setiap variant untuk free GPU memory!

# ┌──────────────────────────────────────────────────────┐
VARIANT             = 'small'   # ganti: 'small' | 'base' | 'large'
USE_GRAD_CHECKPOINT = False     # set True untuk Large di V100 < 32GB
# └──────────────────────────────────────────────────────┘

from mmengine.runner import Runner

output_dir = f'{PROJECT_ROOT}/artifacts/weights/vitpose/{VARIANT}'
os.makedirs(output_dir, exist_ok=True)

print(f'Training ViTPose-{VARIANT.capitalize()}')
print(f'Output: {output_dir}')

cfg = make_vitpose_config(VARIANT, COCO_ANN_DIR, IMAGES_DIR, output_dir, USE_GRAD_CHECKPOINT)

t_start = datetime.now()
runner  = Runner.from_cfg(cfg)
runner.train()
t_end = datetime.now()
duration = (t_end - t_start).total_seconds() / 3600

log = {
    'model_family': 'vitpose', 'model_variant': VARIANT,
    'trained_at': t_start.isoformat(), 'duration_hours': round(duration, 2),
    'grad_checkpoint': USE_GRAD_CHECKPOINT,
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
    'pretrained_from': VITPOSE_PRETRAINED[VARIANT],
}
with open(f'{output_dir}/train_log.json', 'w') as f:
    json.dump(log, f, indent=2)

print(f'\nSelesai! Duration: {duration:.1f}h')
del runner; gc.collect(); torch.cuda.empty_cache()
print('GPU memory cleared — restart runtime sebelum train variant berikutnya')

In [ ]:
# Cell 6: Simpan config ke artifacts (dibutuhkan saat inference)
import shutil, glob as gl
for v in ['small','base','large']:
    out = f'{PROJECT_ROOT}/artifacts/weights/vitpose/{v}'
    src = f'{CONFIG_DIR}/td-hm_ViTPose-{v}_8xb64-210e_coco-256x192.py'
    dst = f'{out}/vitpose_{v}_config.py'
    if os.path.exists(out) and os.path.exists(src):
        shutil.copy(src, dst); print(f'  Config saved: vitpose_{v}_config.py')
    else: print(f'  Skip {v} — belum selesai atau config tidak ada')
print('\nStatus weights:')
for v in ['small','base','large']:
    d = f'{PROJECT_ROOT}/artifacts/weights/vitpose/{v}'
    best = gl.glob(f'{d}/best*.pth')
    if best:
        f = max(best, key=os.path.getmtime)
        print(f'  {v}: {os.path.basename(f)} ({os.path.getsize(f)/1024**2:.0f} MB)')
    else: print(f'  {v}: belum selesai')
print('\nLangkah selanjutnya: 02d_train_detr.ipynb')